# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We will examine the Croissant schema, inspect available record sets and fields (by `@id`), and perform basic processing.

### Dataset Source
The dataset schema is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and preview dataset info using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL for the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
# The dataset.metadata object wraps all metadata/info
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review all available record sets (by `@id`), their columns/fields and field IDs. All entities are shown by their Croissant `@id` values.

In [ ]:
# List available record sets by their @id and display summary info
print("Available Record Sets (by @id):\n")
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in this dataset (check the schema or dataset definition).")
else:
    for rset in record_sets:
        print(f"- Record set @id: {rset.id}\n  Name: {rset.name if hasattr(rset, 'name') else ''}")
        if rset.fields:
            print("  Fields (columns):")
            for fld in rset.fields:
                print(f"    - {fld.id} ({fld.name if hasattr(fld, 'name') else ''})")
        print("")

## 3. Data Extraction
Let's load data from the main record set into a DataFrame for exploration. For each entity, we use IDs for unambiguous reference.

**Note:** If the dataset contains multiple record sets, they'll be loaded.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in getattr(dataset.metadata, "record_sets", [])]

# Prepare DataFrames for all record sets
dataframes = {}
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded record set: {rset_id} (rows: {len(df)}, columns: {list(df.columns)})")
    except Exception as e:
        print(f"Could not load record set {rset_id}: {e}")

# Show the main DataFrame (pick the first available set)
if dataframes:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No records could be loaded.")

## 4. Exploratory Data Analysis (EDA)
We will demonstrate filtering, normalization, and grouping operations using field `@id`s. Adjust `numeric_field_id` and `group_field_id` below as appropriate (refer to the output above to choose actual IDs present in the dataset).

In [ ]:
# Example: Select a numeric field and group field by their Croissant @id (or column name).
# Please refer to the dataframes[main_record_set_id].columns for valid field IDs.

# Replace these with the actual field IDs from the dataset, e.g.:
# numeric_field_id = 'cr:protein_weight'         # Example only
# group_field_id = 'cr:sample_condition'         # Example only
# Update with valid field id strings, e.g. from the displayed columns list.

if dataframes:
    # By default, show all columns for manual selection
    print("Main DataFrame columns (use these @id as field keys):")
    print(list(dataframes[main_record_set_id].columns))
    
    # You can set these to appropriate values found in the dataset for demo
    numeric_field_id = None
    for col in dataframes[main_record_set_id].columns:
        # Heuristically pick a likely numeric column
        if dataframes[main_record_set_id][col].dtype in ['int64', 'float64']:
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Try again: pick the first column as fallback
        numeric_field_id = dataframes[main_record_set_id].columns[0]
    print(f"Using numeric field: {numeric_field_id}\n")

    # Try to choose a group field (categorical)
    group_field_id = None
    for col in dataframes[main_record_set_id].columns:
        if dataframes[main_record_set_id][col].dtype == 'object' and col != numeric_field_id:
            group_field_id = col
            break
    print(f"Using group field: {group_field_id}\n")

    # Now process: filter, normalize, group
    threshold = dataframes[main_record_set_id][numeric_field_id].mean() if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][numeric_field_id]) else 0
    # Filter for illustration
    filtered_df = dataframes[main_record_set_id]
    try:
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    except Exception:
        pass
    print(f"Filtered DataFrame where {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} rows):\n")
    display(filtered_df.head())

    # Normalize the numeric field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the chosen group field (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped (by mean) by field: {group_field_id}")
        display(grouped_df.reset_index().head())
else:
    print("No data available for EDA.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and a grouped bar plot if a group field exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes:
    # Simple histogram of numeric field
    plt.figure(figsize=(7, 4))
    df = dataframes[main_record_set_id]
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        sns.histplot(df[numeric_field_id].dropna(), bins=30)
        plt.title(f"Histogram of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    
    # Boxplot by group if available
    if group_field_id and group_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
We demonstrated how to access and explore a FAIR-compliant Croissant dataset using `mlcroissant`, referencing all entities by their `@id`. Further in-depth analysis can be performed by selecting relevant fields using their Croissant IDs and applying custom domain knowledge.